In [13]:
%pip install torch torchvision torchaudio ftfy regex tqdm pillow opencv-python open_clip_torch pillow
%pip install git+https://github.com/mlfoundations/open_clip.git

  Cloning https://github.com/mlfoundations/open_clip.git to /tmp/pip-req-build-t_xxtn0b
  Running command git clone --filter=blob:none --quiet https://github.com/mlfoundations/open_clip.git /tmp/pip-req-build-t_xxtn0b
  Resolved https://github.com/mlfoundations/open_clip.git to commit a87f11eaf354000d2736580855ae0d9b76ad2a22
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [25]:
import cv2
import os
import torch
import numpy as np
from tqdm import tqdm
import open_clip
from PIL import Image

In [26]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model, _, preprocess = open_clip.create_model_and_transforms(
    "ViT-L-14", pretrained="openai", quick_gelu=True  # bật quickgelu
)
model = model.to(device)
model.eval()


CLIP(
  (visual): VisionTransformer(
    (conv1): Conv2d(3, 1024, kernel_size=(14, 14), stride=(14, 14), bias=False)
    (patch_dropout): Identity()
    (ln_pre): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
    (transformer): Transformer(
      (resblocks): ModuleList(
        (0-23): 24 x ResidualAttentionBlock(
          (ln_1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=1024, out_features=1024, bias=True)
          )
          (ls_1): Identity()
          (ln_2): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (mlp): Sequential(
            (c_fc): Linear(in_features=1024, out_features=4096, bias=True)
            (gelu): QuickGELU()
            (c_proj): Linear(in_features=4096, out_features=1024, bias=True)
          )
          (ls_2): Identity()
        )
      )
    )
    (ln_post): LayerNorm((1024,), eps=1e-05, elementwise_affine=Tru

In [ ]:
def get_embedding(image: Image.Image):
    image_input = preprocess(image).unsqueeze(0).to(device)
    with torch.no_grad():
        embedding = model.encode_image(image_input)
        embedding /= embedding.norm(dim=-1, keepdim=True)
    return embedding.cpu().numpy().flatten()

In [28]:
def euclidean_distance(a, b):
    return np.linalg.norm(a - b)

In [ ]:
def extract_and_save_keyframes(video_path, output_dir, threshold=0.6, step=6):
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    keyframes = []
    last_embedding = None
    frame_idx = 0

    while frame_idx < total_frames:
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        ret, frame = cap.read()
        if not ret:
            break

        image = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        emb = get_embedding(image)

        if last_embedding is None or euclidean_distance(emb, last_embedding) > threshold:
            keyframes.append(frame_idx)
            last_embedding = emb

            output_path = os.path.join(output_dir, f"{frame_idx}.webp")
            image.save(output_path, format="WEBP", lossless=False, quality=95, method=4)
            print(f"✅ Extracted keyframe at frame {frame_idx}")

        frame_idx += step

    cap.release()
    return keyframes


In [30]:
video_path = "L01_V001.mp4"
os.path.exists(video_path)

True

In [ ]:
output_dir = "L01_V001"
threshold = 0.4
step = 6

keyframe_indices = extract_and_save_keyframes(video_path, output_dir, threshold, step)
print(f"Extracted {len(keyframe_indices)} keyframe(s).")

✅ Extracted keyframe at frame 0
✅ Extracted keyframe at frame 3
✅ Extracted keyframe at frame 5
✅ Extracted keyframe at frame 7
✅ Extracted keyframe at frame 8
✅ Extracted keyframe at frame 9
✅ Extracted keyframe at frame 15
✅ Extracted keyframe at frame 24
✅ Extracted keyframe at frame 40
✅ Extracted keyframe at frame 48
✅ Extracted keyframe at frame 53
✅ Extracted keyframe at frame 56
✅ Extracted keyframe at frame 91
✅ Extracted keyframe at frame 108
✅ Extracted keyframe at frame 115
✅ Extracted keyframe at frame 125
✅ Extracted keyframe at frame 138
✅ Extracted keyframe at frame 167
✅ Extracted keyframe at frame 307
✅ Extracted keyframe at frame 412
✅ Extracted keyframe at frame 414
✅ Extracted keyframe at frame 415
✅ Extracted keyframe at frame 421
✅ Extracted keyframe at frame 423
✅ Extracted keyframe at frame 431
✅ Extracted keyframe at frame 436
✅ Extracted keyframe at frame 439
✅ Extracted keyframe at frame 440
✅ Extracted keyframe at frame 441
✅ Extracted keyframe at frame 442